Link word embedding: https://fasttext.cc/docs/en/crawl-vectors.html


In [ ]:
import gensim
import numpy as np
from gensim.models import KeyedVectors
from gensim.models.fasttext import load_facebook_vectors

# Chỉ load một số lượng từ phổ biến nhất định (VD: 200,000 từ) để tiết kiệm RAM
# Tải bộ FastText tiếng Anh và tiếng Việt (cc.en.300.vec và cc.vi.300.vec)

en_emb = load_facebook_vectors(r"E:\07_models\cc.en.300.bin")
vi_emb = load_facebook_vectors(r"E:\07_models\cc.vi.300.bin")
# chạy 5 phút

cách 2 dùng thư viện module fasttext

In [ ]:
# en_emb = fasttext.load_model(r"E:\07_models\cc.en.300.bin")
# vi_emb = fasttext.load_model(r"E:\07_models\cc.vi.300.bin")

In [14]:
def load_word_pairs(filename):
    en_vi_pairs = []
    en_vectors = []
    vi_vectors = []
    
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            en, vi = line.rstrip().split("\t")
            # Bỏ qua nếu từ không có trong tập embeddings
            if en not in en_emb or vi not in vi_emb:
                continue
            en_vi_pairs.append((en, vi))
            en_vectors.append(en_emb[en])
            vi_vectors.append(vi_emb[vi])
            
    return en_vi_pairs, np.array(en_vectors), np.array(vi_vectors)

# X_train chứa các vector tiếng Anh, Y_train chứa các vector tiếng Việt
en_vi_train, X_train, Y_train = load_word_pairs(r"E:\01_Datasets\en-vi.0-5000.txt")
en_vi_test, X_test, Y_test = load_word_pairs(r"E:\01_Datasets\en-vi.5000-6500.txt")

In [15]:
def learn_transform(X, Y):
    # Bước 1: Tính ma trận M = X^T * Y
    M = np.dot(X.T, Y)
    
    # Bước 2: Thực hiện phân tích SVD
    # Hàm np.linalg.svd trả về 3 thành phần: U, S (tức là Sigma), và V^T
    U, S, Vt = np.linalg.svd(M)
    
    # Bước 3: Tính ma trận trực giao tối ưu W* = U * V^T
    W_star = np.dot(U, Vt)
    
    return W_star

# Học không gian biến đổi W
W = learn_transform(X_train, Y_train)

In [20]:
def translate(sentence):
    """
    Dịch từng từ trong câu từ tiếng Anh sang tiếng Việt
    """
    words = sentence.split()
    translated_words = []
    
    for word in words:
        # Kiểm tra từ có tồn tại trong từ điển và là ký tự hay không
        if word in en_emb and word.isalpha():
            en_vec = en_emb[word]
            
            # Nhân với ma trận biến đổi W để chuyển sang không gian tiếng Việt
            mapped_vec = np.matmul(en_vec, W)
            
            # Tìm từ tiếng Việt gần nhất (top 1) trong không gian embedding
            best_match = vi_emb.most_similar(positive=[mapped_vec], topn=1)[0][0]
            translated_words.append(best_match)
        else:
            # Nếu không tìm thấy, giữ nguyên từ gốc
            translated_words.append(word)
            
    return " ".join(translated_words)

# Chạy thử thuật toán
source_sentence = "Hello everybody."
print("Nguồn:", source_sentence)
print("Dịch:", translate(source_sentence))

Nguồn: Hello everybody.
Dịch: hi_all everybody.


Kết luận: ngữ cảnh tiếng Việt và Anh cần thuật toán tối ưu hơn

1. Nguyên nhân chính khiến kết quả dịch saiThiếu ngữ cảnh (Context-free): Thuật toán của bạn dịch từng từ một cách độc lập (for word in words:). Ngôn ngữ tự nhiên phụ thuộc rất nhiều vào ngữ cảnh (ví dụ: từ "bank" có thể là "ngân hàng" hoặc "bờ sông"). Phương pháp này không hiểu được mối quan hệ giữa các từ trong câu.  Sai lệch không gian Vector (Alignment issues): Việc dùng SVD để tìm ma trận xoay $W$ chỉ khớp các không gian vector một cách tuyến tính. Tuy nhiên, phân bố từ vựng trong tiếng Anh và tiếng Việt không đồng nhất (ví dụ: tiếng Việt có nhiều từ ghép, trật tự từ khác nhau).  Vấn đề từ đa nghĩa và từ tương đương: Hàm vi_emb.most_similar sẽ trả về từ tiếng Việt gần nhất về mặt hình học trong không gian vector. Nếu từ tiếng Anh đó không có bản dịch sát nghĩa hoặc vector của nó trong tiếng Việt nằm gần một từ khác có nghĩa khác, máy sẽ dịch sai.  Dữ liệu huấn luyện: Tập dữ liệu en-vi.0-5000.txt có thể chưa đủ phong phú để mô hình học được các cách diễn đạt phức tạp hoặc các biến thể từ vựng
2. Cách cải thiệnNếu bạn muốn kết quả tốt hơn, hãy cân nhắc các hướng đi sau:Sử dụng mô hình dịch máy Transformer (Neural Machine Translation): Phương pháp ánh xạ vector từ (như bạn đang làm) đã cũ. Hiện nay, các mô hình như Transformer (có sẵn trong thư viện HuggingFace Transformers) xử lý toàn bộ câu để hiểu ngữ cảnh trước khi dịch.Xử lý tiền mã hóa (Preprocessing): Trong hàm translate, bạn đang bỏ qua dấu câu và chữ viết hoa (word.isalpha()). Khi gặp dấu phẩy hoặc từ viết hoa, nó sẽ không dịch. Bạn nên tách dấu câu ra khỏi từ trước khi đưa vào mô hình.  Sử dụng kỹ thuật tinh chỉnh (Refinement): Trong học máy cho dịch thuật không giám sát, sau khi có ma trận $W$ ban đầu, người ta thường dùng thuật toán Procrustes Refinement (lặp lại việc tìm $W$ dựa trên các cặp từ tốt nhất đã dịch được) để tăng độ chính xác.Cân nhắc về "Dịch từng từ": Với các ngôn ngữ có cấu trúc khác xa nhau như Anh-Việt, việc dịch từng từ hầu như không thể tạo ra câu tự nhiên. Nếu bạn bắt buộc phải dùng Word Embedding, hãy cân nhắc sử dụng các thư viện như BERT hoặc mBERT (Multilingual BERT) để trích xuất đặc trưng ngữ cảnh thay vì dùng FastText tĩnh.